# Booking.com Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/booking_reviews.py`.

Covers:
1. Web scraping – fetching reviews from Booking.com hotel pages via Playwright
2. Ollama classification – topic & sentiment detection
3. Manual review input – adding reviews that the scraper can't fetch

**Requirements:** Ollama running locally with `mistral:7b`.
No API key needed – Booking.com reviews are scraped from the public website.

**Note:** Booking.com renders reviews client-side via JavaScript. The scraper
uses Playwright (headless Chromium) to render the page and extract reviews.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [ ]:
import importlib
import src.sites.booking_reviews
importlib.reload(src.sites.booking_reviews)

from src.sites import booking_reviews as br

hotel_url = br.BOOKING_URLS.get(br.ANANEA_HOTEL)
print(f'Hotel: {br.ANANEA_HOTEL}')
print(f'Booking URL: {hotel_url}')

## 1. Web Scraping – Single Page

In [ ]:
# Fetch page 1 of reviews from Booking.com using Playwright
pages_html = br.fetch_reviews(hotel_url, max_pages=1, timeout=30000)
if pages_html:
    html = pages_html[0]
    print(f'Content length: {len(html)} chars')
    page_reviews = br._scrape_reviews_from_html(html)
    print(f'Reviews found on page 1: {len(page_reviews)}')
    print()
    for r in page_reviews:
        rating = r.get('rating') or '?'
        print(f"  [{r.get('id', 'N/A')[:12]}] {rating}/10 – {r.get('title', '(no title)')[:50]} ({r.get('published_date', '')})")
else:
    print('Failed to fetch page (anti-bot block?)')

## 2. Web Scraping – All Pages with Pagination

In [4]:
# Fetch reviews across multiple pages
all_reviews = br.booking_get_reviews(hotel_url, max_pages=10, min_delay=2.0, max_delay=4.0)
print(f'Total unique reviews fetched: {len(all_reviews)}')
print()
for r in all_reviews[:10]:
    rating = r.get('rating') or '?'
    print(f"  [{r.get('id', 'N/A')[:12]}] {rating}/10 – {r.get('title', '(no title)')[:50]} ({r.get('published_date', '')})")
if len(all_reviews) > 10:
    print(f'  ... and {len(all_reviews) - 10} more')

Total unique reviews fetched: 92

  [4f6358eef9c4] 10.0/10 – Classy Hotel with all the facilities you need. (2025-11-09)
  [005c09b1a578] 9.0/10 – I would go back again (2025-11-02)
  [d5d4e72d7486] 10.0/10 – Practical great value hotel (2025-10-30)
  [641011773c08] 10.0/10 – Exceptional (2025-10-27)
  [ab0e3aaf1356] 10.0/10 – Excellent stay, would come back again. (2025-10-25)
  [180517ded13c] 9.0/10 – Great addition to the area (2025-10-23)
  [26da3e3d2a56] 10.0/10 – Absolutely the best time. (2025-10-22)
  [89a7b741eeab] 8.0/10 – Relaxing (2025-10-17)
  [88dd32e75913] 10.0/10 – Pleasantly Surprised but happy I discovered and st (2025-10-10)
  [2a3a144f31c3] 10.0/10 – Great stay (2025-10-02)
  ... and 82 more


In [5]:
# Inspect raw scraped data for first review
if all_reviews:
    print(json.dumps(all_reviews[0], indent=2, ensure_ascii=False))

{
  "id": "4f6358eef9c41bc1",
  "rating": 10.0,
  "title": "Classy Hotel with all the facilities you need.",
  "text": "Great hotel with very friendly staff.  Rooms are very large, comfortable and well appointed.  All in all an excellent place to spend a few days or longer ....\nEverything was great.",
  "positive_text": "Great hotel with very friendly staff.  Rooms are very large, comfortable and well appointed.  All in all an excellent place to spend a few days or longer ....",
  "negative_text": "Everything was great.",
  "published_date": "2025-11-09",
  "stay_date": "October 2025",
  "room_name": "One-Bedroom Suite",
  "traveler_type": "Couple",
  "num_nights": "2 nights ·",
  "author_name": "Geoffrey Australia",
  "country": ""
}


## 3. Ollama Classification – Test

In [4]:
# Check Ollama availability
import requests
from src.classification import classify_booking_review, is_ollama_available

ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

Ollama available: True
Models: ['mistral:7b']


In [ ]:
# Classify a single review (pick the first with text)
test_review = next((r for r in all_reviews if r.get('positive_text') or r.get('negative_text')), None)
if test_review and ollama_ok:
    print(f"Review: {test_review.get('title', '(no title)')}")
    print(f"Positive: {test_review.get('positive_text', '')[:200]}")
    print(f"Negative: {test_review.get('negative_text', '')[:200]}")
    print()
    topics = classify_booking_review(test_review.get('positive_text', ''), test_review.get('negative_text', ''))
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        detail = f" ({t['detail']})" if t.get('detail') else ''
        print(f"  {icon} {t['topic']} = {t['sentiment']}{detail}")
else:
    print('No review with text found or Ollama not available')

In [ ]:
# Classify ALL fetched reviews using Booking-specific prompt, save to JSON
from datetime import datetime

if ollama_ok:
    json_path = str(ROOT / 'data' / 'booking_reviews.json')
    existing = br.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in all_reviews:
        positive_text = r.get('positive_text', '')
        negative_text = r.get('negative_text', '')
        if not positive_text and not negative_text:
            continue

        topics = classify_booking_review(positive_text, negative_text)

        review_id = str(r.get('id', ''))
        text = r.get('text', '')

        record = {
            'id': review_id,
            'hotel': br.ANANEA_HOTEL,
            'source': 'booking',
            'rating': r.get('rating'),
            'title': r.get('title', ''),
            'text': text,
            'positive_text': positive_text,
            'negative_text': negative_text,
            'published_date': r.get('published_date', ''),
            'stay_date': r.get('stay_date', ''),
            'room_name': r.get('room_name', ''),
            'trip_type': r.get('trip_type', '') or 'Unknown',
            'author_name': r.get('author_name', ''),
            'country': r.get('country', '') or 'Unknown',
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if review_id in existing_by_id else 'added'
        existing_by_id[review_id] = record

        rating = r.get('rating') or 0
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{rating}/10 [{action}] {r.get('title', 'N/A')[:45]:45s} → {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    br.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available – skip classification')

## 4. Manual Review Input

Booking.com may block scraping or return limited results.
Use this section to manually add reviews you found on the website.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'reviewer_name': 'Guest',                       # reviewer name
    'rating': 9.0,                                  # 0-10 (Booking scale)
    'title': 'Great hotel',                         # review title
    'positive_text': 'Paste positive comments...',  # what the guest liked
    'negative_text': 'Paste negative comments...',  # what the guest disliked
    'published_date': '2026-03-01',                 # YYYY-MM-DD
    'stay_date': 'February 2026',                   # month + year
    'room_name': 'Double or Twin Room',             # room type
    'trip_type': 'Couple',                          # Solo, Couple, Family, Group
    'country': 'United Kingdom',                    # reviewer country
}
# ========================================

# Combine text
text_parts = []
if manual_review['positive_text'] and 'Paste' not in manual_review['positive_text']:
    text_parts.append(manual_review['positive_text'])
if manual_review['negative_text'] and 'Paste' not in manual_review['negative_text']:
    text_parts.append(manual_review['negative_text'])
combined_text = '\n'.join(text_parts)

# Generate unique ID
id_seed = f"{manual_review['reviewer_name']}_{manual_review['published_date']}_{manual_review['title']}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

record = {
    'id': review_id,
    'hotel': br.ANANEA_HOTEL,
    'source': 'booking',
    'rating': manual_review['rating'],
    'title': manual_review['title'],
    'text': combined_text,
    'positive_text': manual_review.get('positive_text', ''),
    'negative_text': manual_review.get('negative_text', ''),
    'published_date': manual_review['published_date'],
    'stay_date': manual_review.get('stay_date', ''),
    'room_name': manual_review.get('room_name', ''),
    'trip_type': manual_review.get('trip_type', '') or 'Unknown',
    'author_name': manual_review['reviewer_name'],
    'country': manual_review.get('country', '') or 'Unknown',
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
pos = record.get('positive_text', '')
neg = record.get('negative_text', '')
if ollama_ok and (pos or neg) and 'Paste' not in pos and 'Paste' not in neg:
    topics = classify_booking_review(pos, neg)
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        detail = f" ({t['detail']})" if t.get('detail') else ''
        print(f"  {icon} {t['topic']} = {t['sentiment']}{detail}")
else:
    print('Ollama not available or no text \u2013 skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'booking_reviews.json')
existing = br.load_reviews(json_path)

existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'\u26a0\ufe0f  Review {record["id"]} already exists \u2013 skipping')
else:
    existing.append(record)
    br.save_reviews(existing, json_path)
    print(f'\u2705 Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once.

In [ ]:
import hashlib

batch_reviews = [
    {
        'reviewer_name': 'Traveler',
        'rating': 9.0,
        'title': 'Wonderful stay',
        'positive_text': 'Replace with actual positive text...',
        'negative_text': 'Replace with actual negative text...',
        'published_date': '2026-02-15',
        'stay_date': 'January 2026',
        'room_name': 'Double or Twin Room',
        'trip_type': 'Couple',
        'country': 'United Kingdom',
    },
    # Add more reviews here...
]

json_path = str(ROOT / 'data' / 'booking_reviews.json')
existing = br.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['reviewer_name']}_{mr['published_date']}_{mr['title']}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

    if rid in existing_ids:
        print(f'⚠️  Skip duplicate: {mr["title"]}')
        continue

    pos = mr.get('positive_text', '')
    neg = mr.get('negative_text', '')
    text_parts = []
    if pos and 'Replace with' not in pos:
        text_parts.append(pos)
    if neg and 'Replace with' not in neg:
        text_parts.append(neg)
    combined_text = '\n'.join(text_parts)

    rec = {
        'id': rid,
        'hotel': br.ANANEA_HOTEL,
        'source': 'booking',
        'rating': mr['rating'],
        'title': mr['title'],
        'text': combined_text,
        'positive_text': pos,
        'negative_text': neg,
        'published_date': mr.get('published_date', ''),
        'stay_date': mr.get('stay_date', ''),
        'room_name': mr.get('room_name', ''),
        'trip_type': mr.get('trip_type', '') or 'Unknown',
        'author_name': mr['reviewer_name'],
        'country': mr.get('country', '') or 'Unknown',
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
    }

    has_text = pos and 'Replace with' not in pos or neg and 'Replace with' not in neg
    if ollama_ok and has_text:
        try:
            topics = classify_booking_review(pos, neg)
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')

    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"✅ {mr['title']} → {topic_str or '(unclassified)'}")

if added:
    br.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [ ]:
json_path = str(ROOT / 'data' / 'booking_reviews.json')
reviews = br.load_reviews(json_path)

needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics')
               and (r.get('positive_text') or r.get('negative_text'))]
unclassified = [r for r in reviews if not r.get('classified')
                and (r.get('positive_text') or r.get('negative_text'))]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_booking_review(r.get('positive_text', ''), r.get('negative_text', ''))
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            print(f"  \u2705 {r.get('title', 'N/A')[:40]} \u2192 {topic_str or '(still none)'}")
        except Exception as e:
            print(f"  \u274c {r.get('title', 'N/A')[:40]} \u2192 Error: {e}")

    br.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

## 7. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [5]:
json_path = str(ROOT / 'data' / 'booking_reviews.json')
reviews = br.load_reviews(json_path)

with_text = [r for r in reviews if r.get('positive_text') or r.get('negative_text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_booking_review(r.get('positive_text', ''), r.get('negative_text', ''))
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '\U0001f504' if old_str != new_str else '\u2705'
            print(f"  {changed} [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            print(f"  \u274c [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]} \u2192 Error: {e}")

    br.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

Total reviews: 60
Reviews with text (will reclassify): 60
  ✅ [1/60] Classy Hotel with all the facilities you
  🔄 [2/60] I would go back again
       old: commodities(pos), commodities(neg), employees(pos), comfort(pos), meals(neg)
       new: commodities(neg), meals(neg), employees(pos), comfort(pos)
  🔄 [3/60] Practical great value hotel
       old: commodities(pos), meals(pos), employees(neg)
       new: commodities(pos), meals(neg), employees(neg)
  🔄 [4/60] Exceptional
       old: commodities(pos), meals(neg), quality_price(pos)
       new: cleaning(pos), commodities(pos), comfort(pos), meals(neg)
  🔄 [5/60] Excellent stay, would come back again.
       old: employees(pos), meals(pos), comfort(neg)
       new: employees(pos), meals(pos), comfort(pos), commodities(neg)
  🔄 [6/60] Great addition to the area
       old: employees(pos), commodities(pos), meals(pos), commodities(neg), quality_price(pos)
       new: commodities(pos), meals(pos), employees(pos), commodities(neg)
  ✅ [7/6

## 8. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'booking_reviews.json')
reviews = br.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if "manual" in str(r.get("id", "")))}')
print()

# Rating distribution (0-10 scale)
ratings = [r.get('rating') for r in reviews if r.get('rating') is not None]
if ratings:
    print(f'Average rating: {sum(ratings)/len(ratings):.1f}/10')
    print(f'Rating range: {min(ratings):.1f} - {max(ratings):.1f}')
    print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')